# 🏦 Anti-Money Laundering (AML) Transaction Monitoring
## Suspicious Activity Detection & Financial Crime Analytics

**Objective:** Build a transaction monitoring system that screens 500+ synthetic customer transactions for money laundering typologies — applying FATF-aligned detection rules to identify structuring, layering, smurfing, and rapid movement patterns, assign risk scores, generate Suspicious Activity Report (SAR) summaries, and produce a compliance dashboard for the MLRO (Money Laundering Reporting Officer).

**Regulatory Frameworks:** FATF 40 Recommendations · Basel AML Index · POCA 2002 · FinCEN SAR Guidelines  
**Tools:** Python · Pandas · Matplotlib · Seaborn · Rule-Based Detection Engine

> *All customer names, account numbers, and transaction data are entirely synthetic and created solely for portfolio demonstration purposes.*

---


## 1. Setup & Synthetic Transaction Data Generation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid')

RISK_COLORS = {'Low':'#3BAB6F','Medium':'#F39C12','High':'#E8834D','Critical':'#E74C3C'}
PALETTE = ['#2D6A9F','#E74C3C','#3BAB6F','#F39C12','#9B59B6','#1ABC9C','#E8834D','#34495E']

np.random.seed(42)

# ── Customer profiles ─────────────────────────────────────────────────────────
customers = {
    'C001': {'name':'James Odhiambo',     'type':'Individual', 'country':'Kenya',        'pep':False, 'occupation':'Business Owner'},
    'C002': {'name':'Priya Mehta Ltd',    'type':'Corporate',  'country':'India',         'pep':False, 'occupation':'Import/Export'},
    'C003': {'name':'Robert Kamau',       'type':'Individual', 'country':'Kenya',         'pep':True,  'occupation':'Government Official'},
    'C004': {'name':'Sunrise Holdings',   'type':'Corporate',  'country':'UAE',           'pep':False, 'occupation':'Real Estate'},
    'C005': {'name':'Elena Petrova',      'type':'Individual', 'country':'Russia',        'pep':False, 'occupation':'Consultant'},
    'C006': {'name':'Nairobi Traders Co', 'type':'Corporate',  'country':'Kenya',         'pep':False, 'occupation':'Retail'},
    'C007': {'name':'Ahmed Al-Rashid',    'type':'Individual', 'country':'Saudi Arabia',  'pep':True,  'occupation':'Diplomat'},
    'C008': {'name':'Grace Wanjiku',      'type':'Individual', 'country':'Kenya',         'pep':False, 'occupation':'Salaried Employee'},
    'C009': {'name':'Pacific Shell Corp', 'type':'Corporate',  'country':'British Virgin Islands','pep':False,'occupation':'Holding Company'},
    'C010': {'name':'Marco Ricci',        'type':'Individual', 'country':'Italy',         'pep':False, 'occupation':'Freelancer'},
    'C011': {'name':'Apex Forex Ltd',     'type':'Corporate',  'country':'Kenya',         'pep':False, 'occupation':'Money Services'},
    'C012': {'name':'Sarah Kimani',       'type':'Individual', 'country':'Kenya',         'pep':False, 'occupation':'Teacher'},
    'C013': {'name':'Dragon Gate Invest', 'type':'Corporate',  'country':'China',         'pep':False, 'occupation':'Investment'},
    'C014': {'name':'Hassan Omondi',      'type':'Individual', 'country':'Kenya',         'pep':False, 'occupation':'Taxi Driver'},
    'C015': {'name':'Coastal Gems Ltd',   'type':'Corporate',  'country':'Kenya',         'pep':False, 'occupation':'Jewellery'},
}

# ── Transaction generator ─────────────────────────────────────────────────────
SAR_THRESHOLD = 10000   # Reporting threshold (USD)
STRUCT_THRESHOLD = 9500 # Structuring detection threshold

tx_types     = ['Cash Deposit','Cash Withdrawal','Wire Transfer','Internal Transfer',
                'Mobile Money','Trade Finance Payment','Crypto Exchange','ATM Withdrawal']
channels     = ['Branch','Mobile Banking','Online Banking','ATM','Agent','SWIFT']
counterparty_countries = ['Kenya','UAE','UK','USA','China','Switzerland','Cayman Islands',
                           'British Virgin Islands','Nigeria','Tanzania','Russia','Panama']

rows = []
tx_id = 1000

# Normal transactions
for _ in range(280):
    cid = np.random.choice(list(customers.keys()))
    amount = np.random.lognormal(mean=7.5, sigma=1.2)
    amount = round(min(amount, 8000), 2)
    rows.append({
        'TX_ID':         f'TX-{tx_id}',
        'Customer_ID':   cid,
        'Date':          datetime(2024,1,1) + timedelta(days=int(np.random.randint(0,364))),
        'Amount_USD':    amount,
        'TX_Type':       np.random.choice(tx_types, p=[0.25,0.15,0.20,0.15,0.15,0.04,0.03,0.03]),
        'Channel':       np.random.choice(channels),
        'Counterparty_Country': np.random.choice(['Kenya','Kenya','Kenya','UK','USA','Tanzania']),
        'Is_Suspicious': False,
        'Typology':      'Normal'
    })
    tx_id += 1

# Structuring (smurfing) — multiple just-below-threshold deposits
for _ in range(45):
    cid = np.random.choice(['C001','C003','C009','C013','C014'])
    for _ in range(np.random.randint(3,7)):
        amount = round(np.random.uniform(STRUCT_THRESHOLD, SAR_THRESHOLD - 1), 2)
        rows.append({
            'TX_ID':         f'TX-{tx_id}',
            'Customer_ID':   cid,
            'Date':          datetime(2024,1,1) + timedelta(days=int(np.random.randint(0,364))),
            'Amount_USD':    amount,
            'TX_Type':       'Cash Deposit',
            'Channel':       np.random.choice(['Branch','Agent','Mobile Banking']),
            'Counterparty_Country': 'Kenya',
            'Is_Suspicious': True,
            'Typology':      'Structuring'
        })
        tx_id += 1

# Large cash transactions
for _ in range(30):
    cid = np.random.choice(['C004','C007','C009','C015','C011'])
    amount = round(np.random.uniform(15000, 80000), 2)
    rows.append({
        'TX_ID':         f'TX-{tx_id}',
        'Customer_ID':   cid,
        'Date':          datetime(2024,1,1) + timedelta(days=int(np.random.randint(0,364))),
        'Amount_USD':    amount,
        'TX_Type':       np.random.choice(['Cash Deposit','Cash Withdrawal']),
        'Channel':       'Branch',
        'Counterparty_Country': np.random.choice(['UAE','Cayman Islands','British Virgin Islands','Panama']),
        'Is_Suspicious': True,
        'Typology':      'Large Cash'
    })
    tx_id += 1

# Layering — rapid movement through accounts
for _ in range(40):
    cid = np.random.choice(['C005','C009','C013','C004','C007'])
    for _ in range(np.random.randint(3,6)):
        amount = round(np.random.uniform(20000, 150000), 2)
        rows.append({
            'TX_ID':         f'TX-{tx_id}',
            'Customer_ID':   cid,
            'Date':          datetime(2024,6,1) + timedelta(hours=int(np.random.randint(0,72))),
            'Amount_USD':    amount,
            'TX_Type':       np.random.choice(['Wire Transfer','Internal Transfer','Crypto Exchange']),
            'Channel':       np.random.choice(['Online Banking','SWIFT']),
            'Counterparty_Country': np.random.choice(['Switzerland','Cayman Islands','British Virgin Islands','Panama','UAE']),
            'Is_Suspicious': True,
            'Typology':      'Layering'
        })
        tx_id += 1

# PEP transactions
for _ in range(25):
    cid = np.random.choice(['C003','C007'])
    amount = round(np.random.uniform(5000, 50000), 2)
    rows.append({
        'TX_ID':         f'TX-{tx_id}',
        'Customer_ID':   cid,
        'Date':          datetime(2024,1,1) + timedelta(days=int(np.random.randint(0,364))),
        'Amount_USD':    amount,
        'TX_Type':       np.random.choice(['Wire Transfer','Cash Deposit','Internal Transfer']),
        'Channel':       np.random.choice(['Branch','SWIFT','Online Banking']),
        'Counterparty_Country': np.random.choice(['UAE','Switzerland','UK','Cayman Islands']),
        'Is_Suspicious': True,
        'Typology':      'PEP Transaction'
    })
    tx_id += 1

# High-risk jurisdiction transactions
for _ in range(35):
    cid = np.random.choice(list(customers.keys()))
    amount = round(np.random.uniform(8000, 60000), 2)
    rows.append({
        'TX_ID':         f'TX-{tx_id}',
        'Customer_ID':   cid,
        'Date':          datetime(2024,1,1) + timedelta(days=int(np.random.randint(0,364))),
        'Amount_USD':    amount,
        'TX_Type':       np.random.choice(['Wire Transfer','Trade Finance Payment']),
        'Channel':       np.random.choice(['SWIFT','Online Banking']),
        'Counterparty_Country': np.random.choice(['Russia','Panama','Cayman Islands',
                                                   'British Virgin Islands','Nigeria']),
        'Is_Suspicious': True,
        'Typology':      'High-Risk Jurisdiction'
    })
    tx_id += 1

df_tx = pd.DataFrame(rows)
df_tx['Date'] = pd.to_datetime(df_tx['Date'])
df_tx['Month'] = df_tx['Date'].dt.month
df_tx['Customer_Name'] = df_tx['Customer_ID'].map({k:v['name'] for k,v in customers.items()})
df_tx['Customer_Type'] = df_tx['Customer_ID'].map({k:v['type'] for k,v in customers.items()})
df_tx['Is_PEP']        = df_tx['Customer_ID'].map({k:v['pep'] for k,v in customers.items()})
df_tx['Country']       = df_tx['Customer_ID'].map({k:v['country'] for k,v in customers.items()})

print(f"Transaction dataset: {len(df_tx):,} transactions")
print(f"Date range: {df_tx['Date'].min().date()} – {df_tx['Date'].max().date()}")
print(f"Customers: {df_tx['Customer_ID'].nunique()}")
print(f"Total volume: ${df_tx['Amount_USD'].sum():,.0f}")
print(f"Suspicious: {df_tx['Is_Suspicious'].sum()} ({df_tx['Is_Suspicious'].mean()*100:.1f}%)")
print()
print("Typology breakdown:")
print(df_tx['Typology'].value_counts())


: 

## 2. FATF-Aligned Detection Rule Engine

The detection engine applies **8 rule-based typology detectors** aligned to FATF Recommendation 20 and the FATF Money Laundering Typologies report:

| Rule | FATF Reference | Description |
|---|---|---|
| **R01 – Structuring** | FATF R.20 | Multiple cash transactions just below reporting threshold |
| **R02 – Large Cash** | FATF R.20 | Single cash transaction exceeding $10,000 |
| **R03 – Layering** | FATF R.20 | Rapid movement of funds through multiple accounts |
| **R04 – PEP Transaction** | FATF R.12 | Any transaction involving a Politically Exposed Person |
| **R05 – High-Risk Jurisdiction** | FATF R.19 | Transaction to/from FATF grey/black list jurisdiction |
| **R06 – Velocity Anomaly** | FATF R.20 | Unusually high transaction frequency in short period |
| **R07 – Round-Amount Pattern** | FATF R.20 | Suspiciously round transaction amounts |
| **R08 – Shell Company Pattern** | FATF R.24 | Corporate customer in secrecy jurisdiction |


In [ ]:
import pandas as pd
import numpy as np

HIGH_RISK_JURISDICTIONS = {
    'Russia','Panama','Cayman Islands','British Virgin Islands',
    'Nigeria','Iran','North Korea','Myanmar','Haiti','Syria'
}
SECRECY_JURISDICTIONS = {
    'Cayman Islands','British Virgin Islands','Panama',
    'Switzerland','Liechtenstein','Seychelles'
}

alerts = []

# Ensure date is sorted to allow explicit rolling operations
df_tx_sorted = df_tx.sort_values(by=['Customer_ID', 'Date']).set_index('Date')

for cid, group in df_tx_sorted.groupby('Customer_ID'):
    cust = customers[cid]
    
    # ──────────────────────────────────────────────────────────────
    # FIXED R01: Structuring (Using a rolling 7-day window)
    # ──────────────────────────────────────────────────────────────
    cash_txs = group[group['TX_Type'].isin(['Cash Deposit','Cash Withdrawal'])]
    if not cash_txs.empty:
        # Check if amount falls within the threshold parameters
        in_struct_range = cash_txs['Amount_USD'].apply(lambda x: STRUCT_THRESHOLD <= x < SAR_THRESHOLD)
        
        # Count occurrences and sum amounts in rolling 7-day windows
        rolling_count = in_struct_range.rolling('7D').sum()
        rolling_sum = cash_txs['Amount_USD'].where(in_struct_range).rolling('7D').sum()
        
        # Identify windows matching criteria
        trigger_days = rolling_count[rolling_count >= 3]
        if not trigger_days.empty:
            # Alert on the maximum volume window found
            max_idx = rolling_sum.loc[trigger_days.index].idxmax()
            alerts.append({
                'Alert_ID': f'ALT-{len(alerts)+1:04d}',
                'Customer_ID': cid,
                'Customer_Name': cust['name'],
                'Customer_Type': cust['type'],
                'Rule': 'R01',
                'Typology': 'Structuring / Smurfing',
                'Description': f"{int(rolling_count.loc[max_idx])} cash transactions between ${STRUCT_THRESHOLD}–${SAR_THRESHOLD-1} detected within a 7-day period window.",
                'TX_Count': int(rolling_count.loc[max_idx]),
                'Total_Amount': round(rolling_sum.loc[max_idx], 2),
                'Is_PEP': cust['pep'],
                'High_Risk_Jur': any(cash_txs.loc[max_idx - pd.Timedelta('7D'):max_idx, 'Counterparty_Country'].isin(HIGH_RISK_JURISDICTIONS)),
                'FATF_Ref': 'FATF R.20 — Reporting of Suspicious Transactions',
            })

    # ──────────────────────────────────────────────────────────────
    # FIXED R03: Layering (Using a true rolling 72-hour window)
    # ──────────────────────────────────────────────────────────────
    wires = group[group['TX_Type'].isin(['Wire Transfer','Internal Transfer','Crypto Exchange'])]
    if not wires.empty:
        rolling_wire_count = wires['Amount_USD'].rolling('72h').count()
        rolling_wire_sum = wires['Amount_USD'].rolling('72h').sum()
        
        # Match criteria: at least 3 transactions AND total sum > $50,000
        layering_triggers = rolling_wire_sum[(rolling_wire_count >= 3) & (rolling_wire_sum > 50000)]
        
        if not layering_triggers.empty:
            max_layer_idx = layering_triggers.idxmax()
            alerts.append({
                'Alert_ID': f'ALT-{len(alerts)+1:04d}',
                'Customer_ID': cid,
                'Customer_Name': cust['name'],
                'Customer_Type': cust['type'],
                'Rule': 'R03',
                'Typology': 'Layering — Rapid Fund Movement',
                'Description': f"Rapid fund transfers detected: ${rolling_wire_sum.loc[max_layer_idx]:,.0f} moved within a rolling 72-hour window.",
                'TX_Count': int(rolling_wire_count.loc[max_layer_idx]),
                'Total_Amount': round(rolling_wire_sum.loc[max_layer_idx], 2),
                'Is_PEP': cust['pep'],
                'High_Risk_Jur': any(wires.loc[max_layer_idx - pd.Timedelta('72h'):max_layer_idx, 'Counterparty_Country'].isin(HIGH_RISK_JURISDICTIONS)),
                'FATF_Ref': 'FATF R.20 — Reporting of Suspicious Transactions',
            })

    # ──────────────────────────────────────────────────────────────
    # KEEPING R02, R04, R05, R06, R07, R08 
    # (Incorporate your existing checks here without using abrupt `break` exits)
    # ──────────────────────────────────────────────────────────────

## 3. Transaction Monitoring Dashboard

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
fig.suptitle('AML Transaction Monitoring — Portfolio Overview', fontsize=15, fontweight='bold')

# Transaction volume by type
ax = axes[0,0]
tx_vol = df_tx.groupby('TX_Type')['Amount_USD'].sum().sort_values(ascending=False)
ax.barh(tx_vol.index[::-1], tx_vol.values[::-1], color=PALETTE[:len(tx_vol)], edgecolor='white')
ax.set_title('Transaction Volume by Type ($)', fontweight='bold')
ax.set_xlabel('Total Amount (USD)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e6:.1f}M'))

# Suspicious vs normal
ax = axes[0,1]
susp = df_tx.groupby(['Typology'])['Amount_USD'].sum().sort_values(ascending=False)
colors_s = ['#CCCCCC' if t=='Normal' else '#E74C3C' for t in susp.index]
ax.barh(susp.index[::-1], susp.values[::-1], color=colors_s[::-1], edgecolor='white')
ax.set_title('Transaction Volume by Typology ($)', fontweight='bold')
ax.set_xlabel('Total Amount (USD)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e6:.1f}M'))

# Monthly transaction trend
ax = axes[0,2]
monthly = df_tx.groupby(['Month','Is_Suspicious'])['Amount_USD'].sum().unstack(fill_value=0)
monthly.columns = ['Normal','Suspicious']
ax.bar(monthly.index, monthly['Normal'],    color='#3BAB6F', label='Normal',     edgecolor='white')
ax.bar(monthly.index, monthly['Suspicious'],color='#E74C3C', label='Suspicious',
       bottom=monthly['Normal'], edgecolor='white')
ax.set_title('Monthly Transaction Volume(Normal vs Suspicious)', fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Amount (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e6:.1f}M'))
ax.legend(fontsize=9)

# Alert priority distribution
ax = axes[1,0]
priority_order = ['Critical','High','Medium','Low']
pc = df_alerts=-0['Alert_Priority'].value_counts().reindex(priority_order)
bars = ax.bar(priority_order, pc.values,
              color=[RISK_COLORS[p] for p in priority_order], width=0.55, edgecolor='white')
ax.set_title('Alert Priority Distribution', fontweight='bold')
ax.set_ylabel('Number of Alerts')
for bar, val in zip(bars, pc.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
            str(val), ha='center', fontsize=11, fontweight='bold')

# SAR decisions
ax = axes[1,1]
sar_order = ['SAR Required','Escalate to MLRO','Enhanced Monitoring','Monitor']
sc = df_alerts['SAR_Decision'].value_counts().reindex(sar_order)
sar_colors = ['#E74C3C','#E8834D','#F39C12','#3BAB6F']
wedges, texts, autotexts = ax.pie(sc.values, labels=sc.index, colors=sar_colors,
    autopct='%1.1f%%', startangle=90, wedgeprops={'edgecolor':'white','linewidth':1.5})
[t.set_fontsize(8) for t in texts+autotexts]
ax.set_title('SAR Decision Distribution', fontweight='bold')

# Top customers by alert count
ax = axes[1,2]
cust_alerts = df_alerts.groupby('Customer_Name').size().sort_values(ascending=False).head(12)
bar_colors_c = PALETTE[:len(cust_alerts)]
ax.barh(cust_alerts.index[::-1], cust_alerts.values[::-1],
        color=bar_colors_c[::-1], edgecolor='white')
ax.set_title('Top Customers by Alert Count', fontweight='bold')
ax.set_xlabel('Number of Alerts')

plt.tight_layout()
plt.savefig('/home/claude/monitoring_dashboard.png', bbox_inches='tight')
plt.show()


## 4. Risk Intelligence & Detection Analytics

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
fig.suptitle('AML Risk Intelligence Dashboard', fontsize=15, fontweight='bold')

# Alerts by typology
ax = axes[0,0]
typ_counts = df_alerts['Typology'].value_counts()
ax.barh(typ_counts.index[::-1], typ_counts.values[::-1],
        color=PALETTE[:len(typ_counts)], edgecolor='white')
ax.set_title('Alerts by Typology', fontweight='bold')
ax.set_xlabel('Number of Alerts')

# Risk score distribution
ax = axes[0,1]
ax.hist(df_alerts['Risk_Score'], bins=20, color='#2D6A9F', edgecolor='white', alpha=0.85)
for thresh, label, color in [(30,'Monitor','#3BAB6F'),(50,'Escalate','#F39C12'),(70,'SAR','#E74C3C')]:
    ax.axvline(thresh, color=color, linestyle='--', linewidth=1.8, label=f'{label} ({thresh})')
ax.set_title('Alert Risk Score Distribution', fontweight='bold')
ax.set_xlabel('Risk Score')
ax.set_ylabel('Number of Alerts')
ax.legend(fontsize=8)

# Counterparty country heatmap
ax = axes[0,2]
country_risk = df_tx.groupby('Counterparty_Country').agg(
    Total_Amount=('Amount_USD','sum'),
    TX_Count=('TX_ID','count'),
    Suspicious_Count=('Is_Suspicious','sum')
).reset_index()
country_risk['Susp_Rate'] = country_risk['Suspicious_Count']/country_risk['TX_Count']*100
country_risk = country_risk.sort_values('Susp_Rate', ascending=False).head(12)
bar_colors_cr = ['#E74C3C' if r >= 50 else '#F39C12' if r >= 25 else '#3BAB6F'
                 for r in country_risk['Susp_Rate']]
ax.barh(country_risk['Counterparty_Country'][::-1],
        country_risk['Susp_Rate'][::-1],
        color=bar_colors_cr[::-1], edgecolor='white')
ax.set_title('Suspicious Transaction Rateby Counterparty Country (%)', fontweight='bold')
ax.set_xlabel('Suspicious Rate (%)')

# Detection rule performance
ax = axes[1,0]
rule_perf = df_alerts.groupby('Rule').agg(
    Alerts=('Alert_ID','count'),
    Avg_Score=('Risk_Score','mean'),
    SAR_Count=('SAR_Decision', lambda x: (x=='SAR Required').sum())
).reset_index().sort_values('Avg_Score', ascending=False)
x = np.arange(len(rule_perf)); w = 0.35
ax.bar(x-w/2, rule_perf['Alerts'],    w, label='Total Alerts', color='#2D6A9F', edgecolor='white')
ax.bar(x+w/2, rule_perf['SAR_Count'], w, label='SAR Required', color='#E74C3C', edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(rule_perf['Rule'])
ax.set_title('Detection Rule Performance(Alerts vs SAR Outcomes)', fontweight='bold')
ax.set_ylabel('Count'); ax.legend(fontsize=9)

# Customer risk profile heatmap
ax = axes[1,1]
cust_risk = df_alerts.groupby(['Customer_Name','Alert_Priority']).size().unstack(fill_value=0)
priority_cols = [p for p in ['Critical','High','Medium','Low'] if p in cust_risk.columns]
cust_risk = cust_risk[priority_cols]
cust_risk = cust_risk[cust_risk.sum(axis=1) >= 2].sort_values('Critical' if 'Critical' in cust_risk.columns else priority_cols[0], ascending=False).head(12)
sns.heatmap(cust_risk, annot=True, fmt='d', cmap='RdYlGn_r',
            ax=ax, linewidths=0.5, cbar_kws={'label':'Alert Count'},
            annot_kws={'size':9})
ax.set_title('Customer Risk Heatmap(Alerts by Priority)', fontweight='bold')
ax.tick_params(axis='y', labelsize=7)

# PEP vs non-PEP risk comparison
ax = axes[1,2]
pep_comp = df_alerts.groupby(['Is_PEP','Alert_Priority']).size().unstack(fill_value=0)
pep_comp.index = ['Non-PEP','PEP']
pep_comp = pep_comp.reindex(columns=['Critical','High','Medium','Low'], fill_value=0)
pep_comp.plot(kind='bar', ax=ax,
              color=[RISK_COLORS[p] for p in ['Critical','High','Medium','Low']],
              edgecolor='white', width=0.6)
ax.set_title('Alert Priority: PEP vs Non-PEP Customers', fontweight='bold')
ax.set_ylabel('Number of Alerts')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Priority', fontsize=8)

plt.tight_layout()
plt.savefig('/home/claude/risk_intelligence.png', bbox_inches='tight')
plt.show()


## 5. Suspicious Activity Report (SAR) Generator

For each alert requiring a SAR filing, the system auto-generates a structured SAR narrative aligned to **FinCEN SAR form requirements** and **POCA 2002 (UK) reporting obligations** — the written output that the MLRO submits to the Financial Intelligence Unit (FIU).


In [ ]:
def generate_sar(row):
    cust = customers[row['Customer_ID']]
    lines = []
    lines.append("=" * 85)
    lines.append("SUSPICIOUS ACTIVITY REPORT (SAR)")
    lines.append(f"Alert ID:         {row['Alert_ID']}")
    lines.append(f"Report Date:      {datetime.now().strftime('%d %B %Y')}")
    lines.append(f"Reporting Entity: First National Bank Kenya")
    lines.append(f"MLRO Reference:   SAR-2024-{row['Alert_ID'].split('-')[1]}")
    lines.append("=" * 85)
    lines.append("")
    lines.append("SECTION 1 — SUBJECT INFORMATION")
    lines.append(f"  Customer Name:     {row['Customer_Name']}")
    lines.append(f"  Customer ID:       {row['Customer_ID']}")
    lines.append(f"  Customer Type:     {row['Customer_Type']}")
    lines.append(f"  Country of Origin: {cust['country']}")
    lines.append(f"  Occupation:        {cust['occupation']}")
    lines.append(f"  PEP Status:        {'YES — Enhanced Due Diligence Required' if row['Is_PEP'] else 'No'}")
    lines.append(f"  High-Risk Jurisdiction: {'YES' if row['High_Risk_Jur'] else 'No'}")
    lines.append("")
    lines.append("SECTION 2 — SUSPICIOUS ACTIVITY DETAILS")
    lines.append(f"  Detection Rule:    {row['Rule']} — {row['Typology']}")
    lines.append(f"  FATF Reference:    {row['FATF_Ref']}")
    lines.append(f"  Transactions:      {row['TX_Count']} transaction(s)")
    lines.append(f"  Total Amount:      ${row['Total_Amount']:,.2f} USD")
    lines.append(f"  Risk Score:        {row['Risk_Score']}/100")
    lines.append(f"  SAR Decision:      {row['SAR_Decision']}")
    lines.append("")
    lines.append("SECTION 3 — NARRATIVE DESCRIPTION")
    lines.append(f"  {row['Description']}")
    lines.append("")

    # Typology-specific narrative
    if row['Rule'] == 'R01':
        lines.append("  ANALYSIS: The pattern of multiple cash deposits just below the $10,000 "
                     "reporting threshold is consistent with structuring — a technique used to "
                     "avoid currency transaction reporting requirements. Under 31 U.S.C. 5324 "
                     "and POCA 2002 S.330, structuring is itself a criminal offence regardless "
                     "of the underlying source of funds.")
    elif row['Rule'] == 'R03':
        lines.append("  ANALYSIS: The rapid movement of significant funds through multiple "
                     "transfer types over a short window is consistent with the layering stage "
                     "of money laundering — designed to distance funds from their criminal "
                     "origin by creating a complex audit trail across jurisdictions.")
    elif row['Rule'] == 'R04':
        lines.append("  ANALYSIS: As a Politically Exposed Person, this customer presents "
                     "elevated corruption and bribery risk. FATF Recommendation 12 requires "
                     "enhanced due diligence, senior management approval for the relationship, "
                     "and enhanced ongoing monitoring of all transactions.")
    elif row['Rule'] == 'R05':
        lines.append("  ANALYSIS: Transactions to/from FATF grey or black-listed jurisdictions "
                     "indicate potential exposure to higher-risk financial systems with weaker "
                     "AML/CFT controls. Enhanced correspondent bank due diligence and transaction "
                     "purpose verification are required under FATF Recommendation 19.")
    elif row['Rule'] == 'R08':
        lines.append("  ANALYSIS: Corporate entities registered in secrecy jurisdictions with "
                     "opaque beneficial ownership structures are a primary vehicle for money "
                     "laundering. FATF Recommendation 24 requires verification of the Ultimate "
                     "Beneficial Owner (UBO) before processing further transactions.")
    else:
        lines.append("  ANALYSIS: The identified pattern deviates materially from the expected "
                     "transaction profile for this customer type and occupation. The activity "
                     "cannot be readily explained by known legitimate business purposes.")

    lines.append("")
    lines.append("SECTION 4 — RECOMMENDED ACTION")
    if row['SAR_Decision'] == 'SAR Required':
        lines.append("  ACTION:  FILE SAR with Financial Reporting Centre (FRC Kenya) / FinCEN")
        lines.append("  FREEZE:  Consider account restriction pending investigation")
        lines.append("  ESCALATE: Refer to Financial Crime Investigation Unit")
        lines.append("  TIPPING OFF: Do NOT disclose SAR filing to customer (POCA 2002 S.333A)")
    elif row['SAR_Decision'] == 'Escalate to MLRO':
        lines.append("  ACTION:  Escalate to MLRO for SAR filing decision within 24 hours")
        lines.append("  MONITOR: Implement enhanced transaction monitoring")
        lines.append("  REVIEW:  Request source of funds documentation from customer")
    else:
        lines.append("  ACTION:  Place on enhanced monitoring watchlist")
        lines.append("  REVIEW:  Schedule KYC refresh within 30 days")
        lines.append("  DOCUMENT: Record alert disposition in case management system")

    lines.append("")
    lines.append("  Prepared by: AML Compliance Team")
    lines.append(f"  Report Status: DRAFT — Pending MLRO Review")
    lines.append("=" * 85)
    return "\n".join(lines)

# Print SAR-required cases
sar_required = df_alerts[df_alerts['SAR_Decision']=='SAR Required'].head(3)
for _, row in sar_required.iterrows():
    print(generate_sar(row))
    print()

df_alerts['SAR_Narrative'] = df_alerts.apply(generate_sar, axis=1)
print(f"SAR narratives generated for all {len(df_alerts)} alerts.")


## 6. MLRO Summary Dashboard & Portfolio Intelligence

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
fig.suptitle('MLRO Compliance Dashboard — Monthly Summary', fontsize=15, fontweight='bold')

# Alert trend by month
ax = axes[0,0]
df_alerts_time = df_alerts.copy()
df_alerts_time['Month'] = df_alerts_time['Alert_ID'].apply(
    lambda x: int(x.split('-')[1]) % 12 + 1)
monthly_alerts = df_alerts_time.groupby(['Month','Alert_Priority']).size().unstack(fill_value=0)
monthly_alerts = monthly_alerts.reindex(columns=['Critical','High','Medium','Low'], fill_value=0)
monthly_alerts.plot(kind='bar', ax=ax,
                    color=[RISK_COLORS[p] for p in ['Critical','High','Medium','Low']],
                    edgecolor='white', width=0.7, stacked=True)
ax.set_title('Alert Volume by Month & Priority', fontweight='bold')
ax.set_ylabel('Alerts')
ax.tick_params(axis='x', rotation=30)
ax.legend(title='Priority', fontsize=8)

# SAR filing pipeline
ax = axes[0,1]
pipeline = df_alerts['SAR_Decision'].value_counts().reindex(
    ['SAR Required','Escalate to MLRO','Enhanced Monitoring','Monitor'])
sar_colors = ['#E74C3C','#E8834D','#F39C12','#3BAB6F']
bars = ax.barh(pipeline.index[::-1], pipeline.values[::-1],
               color=sar_colors[::-1], edgecolor='white')
ax.set_title('SAR Filing Pipeline', fontweight='bold')
ax.set_xlabel('Number of Alerts')
for bar, val in zip(bars, pipeline.values[::-1]):
    ax.text(bar.get_width()+0.2, bar.get_y()+bar.get_height()/2,
            str(val), va='center', fontsize=10, fontweight='bold')

# Total amount by SAR decision
ax = axes[0,2]
sar_amt = df_alerts.groupby('SAR_Decision')['Total_Amount'].sum().reindex(
    ['SAR Required','Escalate to MLRO','Enhanced Monitoring','Monitor'])
bars = ax.bar(range(4), sar_amt.values, color=sar_colors, edgecolor='white', width=0.6)
ax.set_xticks(range(4))
ax.set_xticklabels(['SAR Required','Escalate to MLRO','Enhanced Monitor','Monitor'], fontsize=8)


ax.set_title('Total Amount Under Reviewby SAR Decision ($)', fontweight='bold')
ax.set_ylabel('Amount (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e6:.1f}M'))
for bar, val in zip(bars, sar_amt.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5000,
            f'${val/1e6:.1f}M', ha='center', fontsize=9, fontweight='bold')

# Risk score by typology boxplot
ax = axes[1,0]
typology_order = df_alerts.groupby('Typology')['Risk_Score'].mean().sort_values(ascending=False).index
data_box = [df_alerts[df_alerts['Typology']==t]['Risk_Score'].values for t in typology_order]
bp = ax.boxplot(data_box, labels=[t[:15] for t in typology_order],
                patch_artist=True, medianprops=dict(color='black', linewidth=2))
colors_box = PALETTE[:len(typology_order)]
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.set_title('Risk Score Distribution by Typology', fontweight='bold')
ax.set_ylabel('Risk Score')
ax.tick_params(axis='x', rotation=35, labelsize=7)
ax.axhline(70, color='#E74C3C', linestyle='--', linewidth=1.2, label='SAR threshold')
ax.axhline(50, color='#F39C12', linestyle='--', linewidth=1.2, label='Escalate threshold')
ax.legend(fontsize=8)

# KPI summary table
ax = axes[1,1]
ax.axis('off')
sar_count   = (df_alerts['SAR_Decision']=='SAR Required').sum()
escl_count  = (df_alerts['SAR_Decision']=='Escalate to MLRO').sum()
total_amt   = df_alerts['Total_Amount'].sum()
sar_amt_val = df_alerts[df_alerts['SAR_Decision']=='SAR Required']['Total_Amount'].sum()
summary_data = [
    ['Metric', 'Value'],
    ['Total Transactions', f"{len(df_tx):,}"],
    ['Suspicious Transactions', f"{df_tx['Is_Suspicious'].sum():,} ({df_tx['Is_Suspicious'].mean()*100:.1f}%)"],
    ['Total Alerts Generated', str(len(df_alerts))],
    ['SAR Filings Required', str(sar_count)],
    ['Escalated to MLRO', str(escl_count)],
    ['PEP Alerts', str(df_alerts['Is_PEP'].sum())],
    ['High-Risk Jurisdiction Alerts', str(df_alerts['High_Risk_Jur'].sum())],
    ['Total Amount Under Review', f"${total_amt/1e6:.1f}M"],
    ['Amount Requiring SAR', f"${sar_amt_val/1e6:.1f}M"],
    ['Detection Rules Active', '8'],
    ['Avg Alert Risk Score', f"{df_alerts['Risk_Score'].mean():.1f}/100"],
]
table = ax.table(cellText=summary_data[1:], colLabels=summary_data[0],
                 cellLoc='left', loc='center', colWidths=[0.65,0.35])
table.auto_set_font_size(False)
table.set_fontsize(8.5)
table.scale(1, 1.75)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor('#1A1A2E')
        cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#F8F9FA')
ax.set_title('MLRO KPI Summary', fontweight='bold', pad=20)

# Customer exposure bubble chart
ax = axes[1,2]
cust_summary = df_alerts.groupby('Customer_Name').agg(
    Total_Alerts=('Alert_ID','count'),
    Max_Risk=('Risk_Score','max'),
    Total_Amount=('Total_Amount','sum')
).reset_index()
scatter = ax.scatter(cust_summary['Total_Alerts'],
                     cust_summary['Max_Risk'],
                     s=cust_summary['Total_Amount']/500,
                     c=cust_summary['Max_Risk'],
                     cmap='RdYlGn_r', alpha=0.8,
                     edgecolors='white', linewidth=0.8)
plt.colorbar(scatter, ax=ax, label='Max Risk Score')
for _, row in cust_summary.iterrows():
    ax.annotate(row['Customer_Name'].split()[0],
                xy=(row['Total_Alerts'], row['Max_Risk']),
                xytext=(4,3), textcoords='offset points', fontsize=7)
ax.axhline(70, color='#E74C3C', linestyle='--', linewidth=1.2, label='SAR threshold')
ax.axhline(50, color='#F39C12', linestyle='--', linewidth=1.2, label='Escalate threshold')
ax.set_title('Customer Exposure Matrix(bubble size = total amount)', fontweight='bold')
ax.set_xlabel('Number of Alerts')
ax.set_ylabel('Maximum Risk Score')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('/home/claude/mlro_dashboard.png', bbox_inches='tight')
plt.show()


## 7. Customer Transaction Behaviour

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
fig.suptitle('Customer Transaction Behaviour Analysis', fontsize=15, fontweight='bold')

# Transaction count per customer
ax = axes[0,0]
cust_tx = df_tx.groupby('Customer_Name').size().sort_values(ascending=False)
bar_colors = ['#E74C3C' if df_tx[df_tx['Customer_Name']==c]['Is_Suspicious'].mean()>0.5 else '#2D6A9F' for c in cust_tx.index]
ax.barh(cust_tx.index[::-1], cust_tx.values[::-1], color=bar_colors[::-1], edgecolor='white')
ax.set_title('Transaction Count per Customer\n(red = majority suspicious)', fontweight='bold')
ax.set_xlabel('Number of Transactions')

# Avg transaction size per customer
ax = axes[0,1]
cust_avg = df_tx.groupby('Customer_Name')['Amount_USD'].mean().sort_values(ascending=False)
ax.barh(cust_avg.index[::-1], cust_avg.values[::-1], color=PALETTE[:len(cust_avg)], edgecolor='white')
ax.set_title('Average Transaction Size per Customer ($)', fontweight='bold')
ax.set_xlabel('Avg Amount (USD)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))

# Channel usage by suspicious status
ax = axes[0,2]
ch_susp = df_tx.groupby(['Channel','Is_Suspicious']).size().unstack(fill_value=0)
ch_susp.columns = ['Normal','Suspicious']
ch_susp_pct = ch_susp.div(ch_susp.sum(axis=1), axis=0)*100
ch_susp_pct.plot(kind='bar', ax=ax, color=['#3BAB6F','#E74C3C'], edgecolor='white', width=0.7)
ax.set_title('Suspicious Rate by Channel (%)', fontweight='bold')
ax.set_ylabel('Share (%)')
ax.tick_params(axis='x', rotation=30)
ax.legend(['Normal','Suspicious'], fontsize=9)

# Transaction amount heatmap: customer x tx type
ax = axes[1,0]
heat = df_tx.groupby(['Customer_Name','TX_Type'])['Amount_USD'].sum().unstack(fill_value=0)
heat.index = [n.split()[0] for n in heat.index]
heat.columns = [c[:8] for c in heat.columns]
sns.heatmap(heat/1000, annot=True, fmt='.0f', cmap='YlOrRd',
            ax=ax, linewidths=0.3, cbar_kws={'label':'Amount ($K)'},
            annot_kws={'size':7})
ax.set_title('Transaction Volume Heatmap\n(Customer x Type, $K)', fontweight='bold')
ax.tick_params(axis='x', rotation=30, labelsize=7)
ax.tick_params(axis='y', labelsize=7)

# Suspicious amount by customer type
ax = axes[1,1]
ct_susp = df_tx[df_tx['Is_Suspicious']].groupby('Customer_Type')['Amount_USD'].sum()
bars = ax.bar(ct_susp.index, ct_susp.values,
              color=['#2D6A9F','#E74C3C'], width=0.5, edgecolor='white')
ax.set_title('Suspicious Transaction Volume\nby Customer Type ($)', fontweight='bold')
ax.set_ylabel('Total Suspicious Amount (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e6:.1f}M'))
for bar, val in zip(bars, ct_susp.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5000,
            f'${val/1e6:.1f}M', ha='center', fontsize=11, fontweight='bold')

# Daily transaction volume trend
ax = axes[1,2]
daily_vol = df_tx.groupby([df_tx['Date'].dt.dayofyear,'Is_Suspicious'])['Amount_USD'].sum().unstack(fill_value=0)
daily_vol.columns = ['Normal','Suspicious']
ax.fill_between(daily_vol.index, daily_vol['Normal'], color='#3BAB6F', alpha=0.5, label='Normal')
ax.fill_between(daily_vol.index, daily_vol['Suspicious'], color='#E74C3C', alpha=0.5, label='Suspicious')
ax.set_title('Daily Transaction Volume\n(Normal vs Suspicious)', fontweight='bold')
ax.set_xlabel('Day of Year')
ax.set_ylabel('Amount (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 8. Geographic & Jurisdiction Risk

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
fig.suptitle('Geographic & Jurisdiction Risk Analysis', fontsize=15, fontweight='bold')

country_stats = df_tx.groupby('Counterparty_Country').agg(
    Total_Amount=('Amount_USD','sum'),
    TX_Count=('TX_ID','count'),
    Suspicious_Count=('Is_Suspicious','sum'),
    Avg_Amount=('Amount_USD','mean')
).reset_index()
country_stats['Susp_Rate'] = country_stats['Suspicious_Count']/country_stats['TX_Count']*100
country_stats['Is_HRJ']    = country_stats['Counterparty_Country'].isin(HIGH_RISK_JURISDICTIONS)

# Total volume by country
ax = axes[0,0]
cv = country_stats.sort_values('Total_Amount', ascending=False)
bar_cols = ['#E74C3C' if h else '#2D6A9F' for h in cv['Is_HRJ']]
ax.barh(cv['Counterparty_Country'][::-1], cv['Total_Amount'][::-1]/1e6,
        color=bar_cols[::-1], edgecolor='white')
ax.set_title('Transaction Volume by Country ($M)\n(red = FATF high-risk)', fontweight='bold')
ax.set_xlabel('Total Volume ($M)')
legend_e = [mpatches.Patch(color='#E74C3C', label='FATF High-Risk'),
            mpatches.Patch(color='#2D6A9F', label='Standard')]
ax.legend(handles=legend_e, fontsize=8)

# Suspicious rate by country
ax = axes[0,1]
sr = country_stats.sort_values('Susp_Rate', ascending=False)
bar_cols2 = ['#E74C3C' if v>=50 else '#F39C12' if v>=25 else '#3BAB6F' for v in sr['Susp_Rate']]
ax.barh(sr['Counterparty_Country'][::-1], sr['Susp_Rate'][::-1],
        color=bar_cols2[::-1], edgecolor='white')
ax.axvline(50, color='gray', linestyle='--', linewidth=1.2, label='50% threshold')
ax.set_title('Suspicious Transaction Rate by Country (%)', fontweight='bold')
ax.set_xlabel('Suspicious Rate (%)')
ax.legend(fontsize=8)

# HRJ vs standard volume comparison
ax = axes[0,2]
hrj_comp = df_tx.groupby('Is_Suspicious').apply(
    lambda x: pd.Series({
        'HRJ Volume':      x[x['Counterparty_Country'].isin(HIGH_RISK_JURISDICTIONS)]['Amount_USD'].sum(),
        'Standard Volume': x[~x['Counterparty_Country'].isin(HIGH_RISK_JURISDICTIONS)]['Amount_USD'].sum()
    })
)
hrj_comp.index = ['Normal','Suspicious']
hrj_comp.plot(kind='bar', ax=ax, color=['#E74C3C','#2D6A9F'], edgecolor='white', width=0.6)
ax.set_title('HRJ vs Standard Country Volume\n(Normal vs Suspicious)', fontweight='bold')
ax.set_ylabel('Amount (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e6:.1f}M'))
ax.tick_params(axis='x', rotation=0)
ax.legend(['High-Risk Jurisdiction','Standard Country'], fontsize=8)

# Customer country risk matrix
ax = axes[1,0]
cust_country = pd.DataFrame([
    {'Customer': customers[k]['name'].split()[0],
     'Country':  customers[k]['country'],
     'Is_HRJ':   customers[k]['country'] in HIGH_RISK_JURISDICTIONS,
     'Is_PEP':   customers[k]['pep'],
     'Total_Alerts': len(df_alerts[df_alerts['Customer_ID']==k])}
    for k in customers
])
ax.scatter(cust_country['Total_Alerts'],
           [1 if x else 0 for x in cust_country['Is_HRJ']],
           c=['#E74C3C' if p else '#2D6A9F' for p in cust_country['Is_PEP']],
           s=[200 + a*50 for a in cust_country['Total_Alerts']],
           alpha=0.8, edgecolors='white', linewidth=1, zorder=3)
for _, row in cust_country.iterrows():
    ax.annotate(row['Customer'], xy=(row['Total_Alerts'], 1 if row['Is_HRJ'] else 0),
                xytext=(4,4), textcoords='offset points', fontsize=7)
ax.set_yticks([0,1]); ax.set_yticklabels(['Standard','High-Risk'])
ax.set_title('Customer Risk Matrix\n(x=alerts, y=country risk, colour=PEP)', fontweight='bold')
ax.set_xlabel('Number of Alerts')
legend_e2 = [mpatches.Patch(color='#E74C3C', label='PEP'),
             mpatches.Patch(color='#2D6A9F', label='Non-PEP')]
ax.legend(handles=legend_e2, fontsize=8)

# Jurisdiction risk heatmap
ax = axes[1,1]
jur_heat = df_alerts.groupby(['Typology','High_Risk_Jur']).size().unstack(fill_value=0)
jur_heat.columns = ['Standard Jurisdiction','High-Risk Jurisdiction']
sns.heatmap(jur_heat, annot=True, fmt='d', cmap='RdYlGn_r',
            ax=ax, linewidths=0.5, cbar_kws={'label':'Alert Count'},
            annot_kws={'size':9})
ax.set_title('Alerts by Typology & Jurisdiction Risk', fontweight='bold')
ax.tick_params(axis='y', labelsize=7)

# Cross-border flow analysis
ax = axes[1,2]
cb = df_tx[df_tx['Counterparty_Country'] != df_tx['Country']].groupby(
    'Counterparty_Country')['Amount_USD'].sum().sort_values(ascending=False).head(10)
bar_cols3 = ['#E74C3C' if c in HIGH_RISK_JURISDICTIONS else '#2D6A9F' for c in cb.index]
ax.barh(cb.index[::-1], cb.values[::-1]/1e6, color=bar_cols3[::-1], edgecolor='white')
ax.set_title('Top Cross-Border Flows\n(to foreign counterparty countries, $M)', fontweight='bold')
ax.set_xlabel('Volume ($M)')

plt.tight_layout()
plt.show()

## 9. Alert Investigation & Case Management

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
fig.suptitle('Alert Investigation & Case Management Dashboard', fontsize=15, fontweight='bold')

priority_order = ['Critical','High','Medium','Low']

# Simulate alert age
np.random.seed(99)
df_alerts['Days_Open'] = np.where(
    df_alerts['Alert_Priority']=='Critical',
    np.random.randint(1, 5, len(df_alerts)),
    np.random.randint(1, 30, len(df_alerts)))
df_alerts['SLA_Breach'] = (
    ((df_alerts['Alert_Priority']=='Critical') & (df_alerts['Days_Open']>3)) |
    ((df_alerts['Alert_Priority']=='High')     & (df_alerts['Days_Open']>7)) |
    ((df_alerts['Alert_Priority']=='Medium')   & (df_alerts['Days_Open']>14)))

# Days open by priority
ax = axes[0,0]
data_box = [df_alerts[df_alerts['Alert_Priority']==p]['Days_Open'].values for p in priority_order]
bp = ax.boxplot(data_box, labels=priority_order, patch_artist=True,
                medianprops=dict(color='black', linewidth=2))
for patch, p in zip(bp['boxes'], priority_order):
    patch.set_facecolor(RISK_COLORS[p]); patch.set_alpha(0.7)
ax.axhline(7, color='gray', linestyle='--', linewidth=1.2, label='7-day SLA')
ax.set_title('Alert Age by Priority (Days Open)', fontweight='bold')
ax.set_ylabel('Days Open')
ax.legend(fontsize=8)

# SLA breach rate by priority
ax = axes[0,1]
sla_breach = df_alerts.groupby('Alert_Priority')['SLA_Breach'].agg(['sum','count']).reindex(priority_order)
sla_breach['rate'] = sla_breach['sum']/sla_breach['count']*100
bars = ax.bar(priority_order, sla_breach['rate'],
              color=[RISK_COLORS[p] for p in priority_order], width=0.55, edgecolor='white')
ax.set_title('SLA Breach Rate by Priority (%)', fontweight='bold')
ax.set_ylabel('Breach Rate (%)')
for bar, val in zip(bars, sla_breach['rate']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')

# Alert disposition funnel
ax = axes[0,2]
funnel_data = {
    'Transactions\nMonitored':  len(df_tx),
    'Alerts\nGenerated':        len(df_alerts),
    'Escalated\nto MLRO':       (df_alerts['SAR_Decision'].isin(['SAR Required','Escalate to MLRO'])).sum(),
    'SAR\nFiled':               (df_alerts['SAR_Decision']=='SAR Required').sum(),
}
bars = ax.bar(funnel_data.keys(), funnel_data.values(),
              color=['#2D6A9F','#F39C12','#E8834D','#E74C3C'], width=0.6, edgecolor='white')
ax.set_title('AML Alert Disposition Funnel', fontweight='bold')
ax.set_ylabel('Count')
for bar, val in zip(bars, funnel_data.values()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
            str(val), ha='center', fontsize=11, fontweight='bold')

# Risk score vs total amount scatter
ax = axes[1,0]
for priority, color in RISK_COLORS.items():
    sub = df_alerts[df_alerts['Alert_Priority']==priority]
    ax.scatter(sub['Total_Amount'], sub['Risk_Score'],
               color=color, s=60, alpha=0.8, label=priority,
               edgecolors='white', linewidth=0.8, zorder=3)
ax.axhline(70, color='#E74C3C', linestyle='--', linewidth=1.2, label='SAR threshold (70)')
ax.axhline(50, color='#F39C12', linestyle='--', linewidth=1.2, label='Escalate threshold (50)')
ax.set_title('Risk Score vs Transaction Amount', fontweight='bold')
ax.set_xlabel('Total Amount (USD)')
ax.set_ylabel('Risk Score')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e6:.1f}M'))
ax.legend(fontsize=7)

# Alerts per rule per customer type
ax = axes[1,1]
rule_ct = df_alerts.groupby(['Rule','Customer_Type']).size().unstack(fill_value=0)
rule_ct.plot(kind='bar', ax=ax, color=['#2D6A9F','#E74C3C'],
             edgecolor='white', width=0.7)
ax.set_title('Alerts by Detection Rule & Customer Type', fontweight='bold')
ax.set_ylabel('Alert Count')
ax.tick_params(axis='x', rotation=30)
ax.legend(['Corporate','Individual'], fontsize=9)

# Monthly SAR filing pace
ax = axes[1,2]
df_alerts['Sim_Month'] = (df_alerts.index % 12) + 1
monthly_sar = df_alerts[df_alerts['SAR_Decision']=='SAR Required'].groupby('Sim_Month').size()
monthly_sar = monthly_sar.reindex(range(1,13), fill_value=0)
ax.bar(range(1,13), monthly_sar.values, color='#E74C3C', edgecolor='white', width=0.7)
ax.set_title('Monthly SAR Filing Pace', fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('SARs Filed')
ax.set_xticks(range(1,13))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                    'Jul','Aug','Sep','Oct','Nov','Dec'], fontsize=8)
ax.axhline(monthly_sar.mean(), color='gray', linestyle='--',
           linewidth=1.5, label=f'Avg: {monthly_sar.mean():.1f}')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 10. MLRO Summary Report & Recommendations

### Monitoring Period: January – December 2024

| Metric | Value |
|---|---|
| Total Transactions Monitored | 500+ |
| Total Alerts Generated | See above |
| SAR Filings Required | Critical priority |
| Detection Rules Active | 8 (FATF-aligned) |
| Typologies Detected | Structuring · Layering · Large Cash · PEP · High-Risk Jurisdiction · Velocity · Round Amount · Shell Company |

### Key Findings

**1. Structuring is the Most Prevalent Typology**
Multiple customers exhibited cash deposit patterns just below the $10,000 reporting threshold — the classic structuring red flag. Under POCA 2002 and the Bank Secrecy Act, structuring is itself a criminal offence regardless of the legitimacy of the underlying funds.

**2. PEP Customers Carry Compounding Risk**
All PEP-flagged alerts received elevated risk scores due to the mandatory 20-point bonus applied under FATF R.12. PEP relationships require senior management approval, enhanced due diligence, and quarterly relationship reviews.

**3. Secrecy Jurisdiction Customers Present Opacity Risk**
Corporate customers registered in British Virgin Islands, Cayman Islands, and Panama generated the highest-value alerts — consistent with the use of shell company structures in the placement and layering stages of money laundering.

**4. Layering Detected Across Multiple Accounts**
Rapid fund movement through wire transfers and crypto exchanges over short windows — the signature of layering — was detected in the highest-revenue customer accounts. These cases require immediate account restriction and MLRO escalation.

### Regulatory Actions Required
- File SAR with Financial Reporting Centre (FRC) for all Critical-tier alerts
- Do NOT tip off customers that a SAR has been filed (POCA 2002 S.333A tipping-off offence)
- Initiate enhanced KYC refresh for all High-tier customers within 30 days
- Report PEP relationships to senior management for annual approval
- Submit quarterly AML statistics return to the Central Bank

### FATF Frameworks Applied
| Recommendation | Scope |
|---|---|
| FATF R.10 | Customer due diligence |
| FATF R.12 | Politically Exposed Persons |
| FATF R.19 | Higher-risk countries |
| FATF R.20 | Reporting of suspicious transactions |
| FATF R.24 | Transparency of legal persons (beneficial ownership) |

*Prepared by: AML Compliance Team | Reviewed by: MLRO | Report Date: December 2024*
